# Lab 1D1: Query Language in Python

**Time**: ~15 min  
**Environment**: Jupyter kernel in VS Code  

In this exercise you will write SQL-like queries against Azure Cosmos DB using the Python `azure-cosmos` SDK. You will query for filtered results, parameterized queries, and compare point read vs query costs.

The lab follows the same structure as the C# version. Run each cell in order to complete the steps.

In [ ]:
%pip install azure-cosmos azure-identity python-dotenv --quiet

## Step 0: Initialize Connection

Set up the Cosmos client connection to the `WorkshopData/Catalog` container.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import os

ENDPOINT = os.environ.get("COSMOS_ENDPOINT")
DB_NAME = "WorkshopData"
CONT_NAME = "Catalog"

if not ENDPOINT:
    raise RuntimeError("COSMOS_ENDPOINT environment variable is required.")

print(f"  endpoint: {ENDPOINT}")
print(f"  database: {DB_NAME}")
print(f"  container: {CONT_NAME}")
print(f"  connected: {ENDPOINT}{DB_NAME}/{CONT_NAME}")

In [ ]:
from azure.cosmos import CosmosClient, PartitionKey
from azure.identity import DefaultAzureCredential

cred = DefaultAzureCredential()
client = CosmosClient(url=ENDPOINT, credential=cred)
db = client.get_database_client(DB_NAME)
container = db.get_container_client(CONT_NAME)
print(f"Connected to: {ENDPOINT}/{DB_NAME}/{CONT_NAME}")

## Step 1: Seed Data (Prebuilt)

Seed sample grocery items into the container. This data is used in all subsequent queries.

In [ ]:
seed_items = [
    {
        "id": "1", "name": "Apples", "category": "fruit", "price": 1.20, "partitionKey": "grocery",
        "tags": ["organic", "seasonal", "domestic"],
        "nutrition": {"calories": 95, "vitamins": ["A", "C"]},
    },
    {
        "id": "2", "name": "Broccoli", "category": "vegetable", "price": 2.50, "partitionKey": "grocery",
        "tags": ["organic", "fresh"],
        "nutrition": {"calories": 55, "vitamins": ["C", "K", "A"]},
    },
    {
        "id": "3", "name": "Bananas", "category": "fruit", "price": 0.80, "partitionKey": "grocery",
        "tags": ["imported", "ripe"],
        "nutrition": {"calories": 105, "vitamins": ["B6", "C"]},
    },
    {
        "id": "4", "name": "Carrots", "category": "vegetable", "price": 1.00, "partitionKey": "grocery",
        "tags": ["organic", "root", "fresh"],
        "nutrition": {"calories": 41, "vitamins": ["A", "K"]},
    },
    {
        "id": "5", "name": "Dates", "category": "fruit", "price": 4.00, "partitionKey": "grocery",
        "tags": ["imported", "dried", "premium"],
        "nutrition": {"calories": 280, "vitamins": ["B6", "K"]},
    },
]

for item in seed_items:
    try:
        response = container.upsert_item(body=item)
        print(f"  Upserted: {item['name']}")
    except Exception as ex:
        print(f"  Error: {ex}")

print(f"\nSeeded {len(seed_items)} items")

## Step 2: Query for All Fruits (STUDENT EXERCISE)

Write a SQL query to retrieve all items with `category == "fruit"`.

**Expected output**: 3 items (Apples, Bananas, Dates) with their prices listed, then the total RU charged.

**Hint**: Use `container.query_items(query=..., parameters=[...], enable_cross_partition_query=True)`. Parameters are passed as a list of dicts with `name` and `value` keys.

In [ ]:
category_to_query = "fruit"  # TODO: change this to query a different category

query = f"SELECT * FROM c WHERE c.category = @cat"

fruits = list(container.query_items(
    query=query,
    parameters=[{"name": "@cat", "value": category_to_query}],
    enable_cross_partition_query=True
))

fruit_query_ru = float(container.client_connection.last_response_headers["x-ms-request-charge"])

print("Fruits:")
for item in fruits:
    print(f"  {item['name']}: ${item['price']}")
print(f"RU charged: {fruit_query_ru}")

## Step 3: Point Read vs Query Cost

Fetch the same single item (`id = "1"`) two ways — a point read and a `SELECT * FROM c WHERE c.id = '1'` query — and compare their RU charges. Same logical result, two access patterns, so the RU difference is a fair head-to-head.

In [ ]:
point_read_item = container.read_item(item="1", partition_key="grocery")
point_read_ru = float(container.client_connection.last_response_headers["x-ms-request-charge"])

single_item_query = "SELECT * FROM c WHERE c.id = '1'"
list(container.query_items(
    query=single_item_query,
    enable_cross_partition_query=True
))
single_item_query_ru = float(container.client_connection.last_response_headers["x-ms-request-charge"])

print("Fetching the same single item (id='1') two different ways.\n")
print(f"Point read (1 item by id + partition key): {point_read_ru} RU")
print(f"Query  (SELECT * FROM c WHERE c.id='1'):   {single_item_query_ru} RU")
if point_read_ru > 0:
    print(f"Point read is {single_item_query_ru / point_read_ru:.1f}x cheaper for fetching a single item by id.")

## Step 4: Parameterized Query (STUDENT EXERCISE)

Write a parameterized `SELECT TOP` query to get the most expensive items.

**Expected output**: 3 items sorted by price descending with their names and prices.

**Hint**: Add the `@limit` parameter using the `parameters` argument of `query_items()`. The LIMIT value should be 3.

In [ ]:
limit_val = 3  # TODO: change this to query a different limit

top_query = "SELECT TOP @limit c.name, c.price FROM c ORDER BY c.price DESC"

top_items = list(container.query_items(
    query=top_query,
    parameters=[
        {"name": "@limit", "value": limit_val}
    ],
    enable_cross_partition_query=True
))

top_query_ru = float(container.client_connection.last_response_headers["x-ms-request-charge"])

print(f"Top {limit_val} items by price (descending):")
for item in top_items:
    print(f"  {item['name']}: ${item['price']}")
print(f"RU charged: {top_query_ru}")

## Step 5: JSON Properties + System Functions

Cosmos DB stores items as JSON, so queries can reach into nested objects and arrays directly. This query combines:

- Nested-property access — `c.nutrition.calories`
- The `ARRAY_CONTAINS` system function — to filter on an element of the `tags` array
- The `CONCAT` system function — to reshape data in the projection

See the [system functions reference](https://learn.microsoft.com/azure/cosmos-db/nosql/query/system-functions) for the full list.

In [ ]:
json_query = (
    "SELECT c.name, CONCAT(c.category, ' category') AS category, c.nutrition.calories "
    "FROM c "
    "WHERE ARRAY_CONTAINS(c.tags, 'organic') AND c.nutrition.calories < 100"
)

results = list(container.query_items(
    query=json_query,
    enable_cross_partition_query=True
))

json_query_ru = float(container.client_connection.last_response_headers["x-ms-request-charge"])

print("Low-calorie organic items:")
for item in results:
    print(f"  {item['name']} ({item['category']}): {item['calories']} cal")
print(f"RU charged: {json_query_ru}")

## Step 6: Subquery Over a Nested Array

A [subquery](https://learn.microsoft.com/azure/cosmos-db/nosql/query/subquery) lets a query iterate or aggregate over a nested array inside each document. This one uses `SELECT VALUE COUNT(1) FROM v IN c.nutrition.vitamins` to count the vitamins per item.

In [ ]:
subquery = (
    "SELECT c.name, "
    "       (SELECT VALUE COUNT(1) FROM v IN c.nutrition.vitamins) AS vitaminCount "
    "FROM c "
    "ORDER BY c.name"
)

results = list(container.query_items(
    query=subquery,
    enable_cross_partition_query=True
))

subquery_ru = float(container.client_connection.last_response_headers["x-ms-request-charge"])

print("Vitamin counts per item:")
for item in results:
    print(f"  {item['name']}: {item['vitaminCount']} vitamins")
print(f"RU charged: {subquery_ru}")

## Lab Complete!

You have completed the query language exercise in Python. You:
- Connected to Cosmos DB using `DefaultAzureCredential`
- Seeded sample data with nested objects and arrays
- Ran a filter query using `query_items()` with parameterized input
- Compared point read vs query cost
- Wrote a parameterized query with `TOP`
- Queried nested JSON properties with `ARRAY_CONTAINS` and `UPPER`
- Wrote a subquery that aggregates a nested array

To run the lab again from scratch, re-run each cell in order.